# ProntoQA Symbolic Pipeline

End-to-end pipeline using real ProntoQA problems from HuggingFace:
1. Config
2. Mount Google Drive
3. Clone repo + install deps
4. Load ProntoQA + generate Qwen traces
5. Train SSAE Phase 1
6. Generate probe data (encode traces through SSAE)
7. Train step-correctness probe
8. Download results

## 0 — Config

In [17]:
# ── Edit these ────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/djaxchi/CoT-checker.git"

# Data generation
# ProntoQA has 500 problems (~48% False) — None = use all, or set e.g. 50 for a smoke test
MAX_PROBLEMS  = None
GEN_BATCH     = 16      # LLM generation batch size

# SSAE training
TRAIN_EPOCHS  = 1
TRAIN_BATCH   = 4
GRAD_ACCUM    = 16       # effective batch = 16 * 4 = 64

# Probe training
PROBE_EPOCHS  = 30
# ── End config ────────────────────────────────────────────────────────────────

import torch
gpu  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.0f} GB")

FREEZE_BACKBONES = vram < 35
print(f"Freeze backbones: {FREEZE_BACKBONES}  ({'T4/V100' if FREEZE_BACKBONES else 'A100'})"
)

GPU : Tesla T4
VRAM: 16 GB
Freeze backbones: True  (T4/V100)


## 1 — Mount Google Drive

Everything is persisted to `MyDrive/CoT-checker/` so runtime resets don't lose progress.

In [11]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT        = "/content/drive/MyDrive/CoT-checker"
DRIVE_DATA        = f"{DRIVE_ROOT}/data"
DRIVE_CKPT        = f"{DRIVE_ROOT}/results/checkpoints"
DRIVE_PROBE_DATA  = f"{DRIVE_ROOT}/results/probe_data"
DRIVE_PROBE_MODEL = f"{DRIVE_ROOT}/results/probes"

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_PROBE_DATA, DRIVE_PROBE_MODEL]:
    os.makedirs(d, exist_ok=True)

PROBLEMS_PATH    = f"{DRIVE_DATA}/prontoqa_hf.jsonl"
TRACES_PATH      = f"{DRIVE_DATA}/prontoqa_traces.jsonl"
CKPT_PATH        = f"{DRIVE_CKPT}/ssae_symbolic_p1.pt"
ENC_PATH         = f"{DRIVE_CKPT}/ssae_symbolic_p1.enc.pt"
PROBE_DATA_PATH  = f"{DRIVE_PROBE_DATA}/symbolic_ssae_p1.npz"
PROBE_MODEL_PATH = f"{DRIVE_PROBE_MODEL}/correctness_probe_symbolic.pt"

print(f"Traces on Drive    : {'YES - skipping generation' if os.path.exists(TRACES_PATH) else 'NO - will generate'}")
print(f"SSAE ckpt on Drive : {'YES - skipping training'   if os.path.exists(CKPT_PATH)   else 'NO - will train'}")
print(f"Probe data on Drive: {'YES - skipping encoding'   if os.path.exists(PROBE_DATA_PATH) else 'NO - will encode'}")
print(f"Probe model on Drive: {'YES' if os.path.exists(PROBE_MODEL_PATH) else 'NO'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Traces on Drive    : NO - will generate
SSAE ckpt on Drive : NO - will train
Probe data on Drive: NO - will encode
Probe model on Drive: NO


## 2 — Clone repo + install dependencies

In [12]:
import os                                                                                                                                                                                                 
os.remove(TRACES_PATH)                                                                                                                                                                                  
os.remove(CKPT_PATH)
print("Deleted. Re-run §3 and §4.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/CoT-checker/data/prontoqa_traces.jsonl'

In [13]:
import os

if not os.path.isdir("/content/CoT-checker"):
    !git clone {GITHUB_REPO} /content/CoT-checker
else:
    !git -C /content/CoT-checker pull

%cd /content/CoT-checker
!pip install -q transformers datasets tqdm

Already up to date.
/content/CoT-checker


## 3 — Load ProntoQA + generate model traces

**Phase A**: loads 500 problems from `renma/ProntoQA` on HuggingFace (~48% False, complex multi-hop chains).  
**Phase B**: runs Qwen2.5-0.5B on each problem to produce model-generated CoT traces.

Skipped automatically if traces already exist on Drive.

In [14]:
import os

if os.path.exists(TRACES_PATH):
    print(f"Traces already exist at {TRACES_PATH} — skipping generation.")
else:
    max_flag = f"--max-problems {MAX_PROBLEMS}" if MAX_PROBLEMS else ""
    !python scripts/generate_symbolic_data.py \
        {max_flag} \
        --problems-out {PROBLEMS_PATH} \
        --traces-out   {TRACES_PATH} \
        --batch-size   {GEN_BATCH} \
        --device cuda
    print("\nDone. Saved to Drive.")

Loading renma/ProntoQA from HuggingFace …
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'renma/ProntoQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 100% 737/737 [00:00<00:00, 4.51MB/s]
ProntoQA_dev_gpt-4.json: 1.01MB [00:00, 116MB/s]
Generating validation split: 100% 500/500 [00:00<00:00, 12044.01 examples/s]
Loaded 500 problems from ProntoQA
True / False split: 258 / 242 (48% False)
Saved 500 problems to /content/drive/MyDrive/CoT-checker/data/prontoqa_hf.jsonl

Loading Qwen2.5-0.5B on cuda …
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 374.59it/s, Materializing param=model.norm.weight]                              
Generating traces: 100% 32/32 [04:05<00:00,  7.67s/it]

Problems with traces : 497/500
Skipped (no steps)   : 3
T

In [15]:
# Sanity check: inspect traces
import json

total_traces = sum(1 for _ in open(TRACES_PATH))
total_steps  = sum(len(json.loads(l)["steps"]) for l in open(TRACES_PATH))
print(f"Traces : {total_traces}")
print(f"Steps  : {total_steps}")
print(f"Avg steps/trace: {total_steps/total_traces:.1f}")
print()
with open(TRACES_PATH) as f:
    for rec in [json.loads(next(f)) for _ in range(3)]:
        print(f"Q: {rec['question'][:120]}")
        print(f"Steps: {rec['steps']}")
        print()

Traces : 497
Steps  : 3736
Avg steps/trace: 7.5

Q: Jompuses are not shy. Jompuses are yumpuses. Each yumpus is aggressive. Each yumpus is a dumpus. Dumpuses are not wooden
Steps: ['Max is a yumpus.', 'Max is a dumpus.', 'Max is a wumpus.', 'Max is a zumpus.', 'Max is a vumpus.', 'Max is a tumpus.', 'Max is a rompus.', 'Max is a zumpus.', 'Max is a vumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.', 'Max is a zumpus.']

Q: Every tumpus is not angry. Tumpuses are rompuses. Every numpus is not bright. Rompuses are not luminous. Rompuses are yu
Steps: ['Stella is a yumpus.', 'Stella is a dumpus.', 'Stella is a vumpus.', 'Stella is a jompus.', 'Stella is a wumpus.', 'Stella is bright.']

Q: Vumpuses are floral. Vumpuses are tumpuses. Tumpuses are brown. Each tumpus is a wumpus. Wumpuses are small. Each wumpus
Steps

## 4 — Train SSAE Phase 1

Trains the sparse autoencoder to reconstruct reasoning steps from sparse latents.  
Backbone frozen on T4 (16 GB), unfrozen on A100 (40 GB).  
Checkpoint saved to Drive — safe to interrupt and resume.

In [18]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  

if os.path.exists(CKPT_PATH):
    print(f"Checkpoint already exists at {CKPT_PATH} — skipping SSAE training.")
else:
    freeze_flag = "--freeze-backbones" if FREEZE_BACKBONES else "--no-freeze-backbones"
    !python scripts/train_ssae_symbolic.py \
        --traces    {TRACES_PATH} \
        --output    {CKPT_PATH} \
        --phase     1 \
        --epochs    {TRAIN_EPOCHS} \
        --batch-size {TRAIN_BATCH} \
        --grad-accum {GRAD_ACCUM} \
        --device    cuda \
        {freeze_flag}

Initialising fresh SSAE (Qwen2.5-0.5B, float32) …
Loading weights: 100% 290/290 [00:00<00:00, 403.58it/s, Materializing param=norm.weight]                              
Loading weights: 100% 290/290 [00:00<00:00, 386.83it/s, Materializing param=norm.weight]                              
Loading weights: 100% 290/290 [00:00<00:00, 406.40it/s, Materializing param=model.norm.weight]                              
  Backbones frozen. Trainable modules: {'projection_mlp'}
  Trainable params: 1,607,424
Loading traces from /content/drive/MyDrive/CoT-checker/data/prontoqa_traces.jsonl …
  Train: 3550 steps  |  Val: 186 steps
  Batches/epoch: 887  |  Phase: 1

Starting Phase 1 training for 1 epoch(s)
  LR=1e-06  L1_weight=1e-04  grad_accum=16
Epoch 1/1: 100% 887/887 [13:19<00:00,  1.11it/s, l1w=1.0e-04, lr=5.2e-07, nll=3.3902, spa=17.234]

Epoch 1 — val_loss_nll: 1.9463  (best: inf)
  [ckpt] saved → /content/drive/MyDrive/CoT-checker/results/checkpoints/ssae_symbolic_p1.pt  (step=55, val_loss=1.

In [19]:
# Verify checkpoint on Drive
import os
for fname in os.listdir(DRIVE_CKPT):
    size = os.path.getsize(f"{DRIVE_CKPT}/{fname}") / 1e6
    print(f"{fname:<50}  {size:6.1f} MB")

ssae_symbolic_p1.enc.pt                                3.2 MB
ssae_symbolic_p1_final.pt                             34.4 MB
ssae_symbolic_p1_final.enc.pt                          3.2 MB
ssae_symbolic_p1.pt                                   34.4 MB


## 5 — Generate probe data

Encodes every `(context, step)` pair through the SSAE encoder → sparse latent `h_c`.  
Labels each step with `PropLogicSolver` (1 = valid deduction, 0 = invalid).  
Saves `(latents, labels)` as `.npz`.

**Check the label distribution here** — target is ~25–45% incorrect steps.

In [20]:
import os

if os.path.exists(PROBE_DATA_PATH):
    print(f"Probe data already exists at {PROBE_DATA_PATH} — skipping encoding.")
else:
    !python scripts/generate_probe_data_symbolic.py \
        --checkpoint {CKPT_PATH} \
        --data       {PROBLEMS_PATH} \
        --output     {PROBE_DATA_PATH} \
        --device     cuda

# Report label distribution
import numpy as np
d = np.load(PROBE_DATA_PATH)
labels = d["correctness"]
pos = int(labels.sum())
neg = len(labels) - pos
print(f"\nProbe data: {len(labels)} steps")
print(f"  Correct (+)  : {pos}  ({pos/len(labels):.1%})")
print(f"  Incorrect (-): {neg}  ({neg/len(labels):.1%})")
print(f"  Majority baseline: {max(pos,neg)/len(labels):.1%}")

Loading Qwen2.5-0.5B for generation …
Loading weights: 100% 290/290 [00:00<00:00, 376.30it/s, Materializing param=model.norm.weight]                              
Loaded 500 problems from /content/drive/MyDrive/CoT-checker/data/prontoqa_hf.jsonl
Generating traces: 100% 500/500 [19:55<00:00,  2.39s/it]

Loading SSAE from /content/drive/MyDrive/CoT-checker/results/checkpoints/ssae_symbolic_p1.pt …
  Slim checkpoint detected (frozen backbones). Re-loading backbone from HF …
Loading weights: 100% 290/290 [00:00<00:00, 416.93it/s, Materializing param=norm.weight]                              
Loading weights: 100% 290/290 [00:00<00:00, 408.40it/s, Materializing param=norm.weight]                              
Loading weights: 100% 290/290 [00:00<00:00, 355.66it/s, Materializing param=model.norm.weight]                              

Problems skipped (no parseable steps): 3
Steps collected: 3208
Correct (+): 1864  (58.1%)
Incorrect (-): 1344  (41.9%)
Majority baseline: 58.1%

Encoding 3208 s

## 6 — Train step-correctness probe

Trains a 3-layer MLP on `(h_c, label)` pairs.  
Target: val accuracy meaningfully above the majority baseline printed above.

In [21]:
!python scripts/train_probe.py \
    --data    {PROBE_DATA_PATH} \
    --output  {PROBE_MODEL_PATH} \
    --epochs  {PROBE_EPOCHS} \
    --device  cuda

Training samples  : 2566
Validation samples: 642
n_latents         : 896
Positive rate     : 0.580  (majority baseline: 0.580)

Epoch  Train Loss   Val Acc
──────────────────────────────
    1      0.7006    0.5872 *
    2      0.6821    0.5872
    3      0.6806    0.5872
    4      0.6862    0.5872
    5      0.6787    0.5872
    6      0.6798    0.5872
    7      0.6766    0.5872
    8      0.6731    0.5872
    9      0.6769    0.6137 *
   10      0.6675    0.5872
   11      0.6619    0.5903
   12      0.6601    0.5872
   13      0.6596    0.6277 *
   14      0.6335    0.5935
   15      0.6574    0.6246
   16      0.6307    0.6012
   17      0.6183    0.6355 *
   18      0.5989    0.5981
   19      0.6192    0.6386 *
   20      0.5941    0.6729 *
   21      0.5809    0.6838 *
   22      0.5770    0.6760
   23      0.5718    0.6838
   24      0.5658    0.6760
   25      0.5656    0.6822
   26      0.5649    0.6916 *
   27      0.5696    0.6807
   28      0.5647    0.6978 *
   29      

## 7 — Download results

Place files locally:
- `ssae_symbolic_p1.enc.pt` → `results/checkpoints/`
- `correctness_probe_symbolic.pt` → `results/probes/`
- `symbolic_ssae_p1.npz` → `results/probe_data/`

In [22]:
from google.colab import files

files.download(ENC_PATH)          # SSAE encoder (~3 MB)
files.download(PROBE_MODEL_PATH)  # correctness probe (~6 MB)
files.download(PROBE_DATA_PATH)   # probe latents + labels .npz

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>